# 과제 10 · 07 · 정비 이력 시스템을 MCP 서버로 열기

## 현업 시나리오
정비 이력 시스템은 설비팀이 운영합니다. 우리 앱 안의 함수가 아니라 **별도 Process** 라고 생각해야 합니다. 그 시스템이 MCP 서버로 Tool 을 열면 우리 Agent 는 import 없이 그 Tool 을 받아 씁니다. 대신 열어 주는 Tool 은 **읽기 전용**이어야 하고, 실패가 어떤 모양으로 오는지 알아야 합니다.

## 학습 목표
- FastMCP 로 서버를 만들고 읽기 전용 목록에 있는 Tool 만 연다.
- `MultiServerMCPClient` 로 Tool 을 받아 오고 그것이 LangChain Tool 인 것을 확인한다.
- MCP 를 거친 실패가 예외가 아니라 결과 글자로 오는 것을 보고, 결과 모양으로 성공을 판단한다.

## 직접 조립
완성된 App을 가져오지 않습니다. 아래에서 작은 fixture와 핵심 함수를 직접 만듭니다.

### 1단계 · 읽기 전용 Tool 만 여는 서버

In [ ]:
# 서버는 별도 파일입니다. 같은 Process 가 아니라는 것이 핵심입니다.
# 서버에는 삭제 함수도 있지만 읽기 전용 목록에 없으므로 열지 않습니다.
# 관찰 포인트: 서버 쪽에는 LangChain 이 없습니다. MCP 는 표준이라 서로를 모릅니다.
import sys
import tempfile
from pathlib import Path

practice_server_code = """
import json
from typing import Literal
from mcp.server.fastmcp import FastMCP

HISTORY = {"HDF": [{"id": "MR-0102", "action": "흡기 필터 교체"}], "TWF": [{"id": "MR-0104", "action": "공구 교체"}]}

def same_type_history(failure_type: Literal["TWF", "HDF", "PWF", "OSF"], before: str) -> str:
    \"\"\"정비 이력에서 이 유형으로 수리한 기록을 찾는다. before 시각보다 앞선 기록만 돌려준다.\"\"\"
    return json.dumps({"records": HISTORY.get(failure_type, [])}, ensure_ascii=False)

def delete_record(record_id: str) -> str:
    \"\"\"기록을 지운다. 이 서버는 이 함수를 열지 않는다.\"\"\"
    return "deleted " + record_id

READ_ONLY = ("same_type_history",)
server = FastMCP("practice-history-readonly")
for fn in (same_type_history, delete_record):
    if fn.__name__ in READ_ONLY:
        server.add_tool(fn)

if __name__ == "__main__":
    server.run()
"""
practice_server_path = Path(tempfile.mkdtemp()) / "practice_history_server.py"
practice_server_path.write_text(practice_server_code, encoding="utf-8")
print("서버 파일:", practice_server_path.name, "· 정의된 함수 2개, 읽기 전용 목록 1개")

**결과 해설** — 서버 파일에는 함수가 둘 있지만 읽기 전용 목록에 있는 `same_type_history` 하나만 서버에 등록합니다. 쓰기 Tool 이 열리지 않는다는 것은 **목록에서 가져다 등록하는 구조**가 보장합니다.

### 2단계 · 붙어서 Tool 받아 오기

In [ ]:
# MultiServerMCPClient 가 서버 Process 를 띄우고 stdio 로 붙어 Tool 목록을 받아 옵니다.
# 세션과 전송을 직접 만들지 않습니다.
# MCP Tool 은 비동기 전용입니다. Jupyter 는 이미 이벤트 루프가 돌고 있어 asyncio.run 을 바로 쓸 수 없으므로,
# 별도 스레드에서 돌립니다(Jupyter 와 일반 Python 양쪽에서 같은 코드가 돕니다).
# Windows 에서 하위 Process(MCP 서버)를 띄우려면 Proactor 루프가 필요해 루프를 직접 고릅니다.
# 관찰 포인트: 받아 온 것은 StructuredTool 이고, Literal 인자는 서버를 건너와도 enum 으로 남습니다.
import asyncio
from concurrent.futures import ThreadPoolExecutor
from langchain_mcp_adapters.client import MultiServerMCPClient

def practice_new_loop():
    return asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.SelectorEventLoop()

def practice_wait(coroutine):
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coroutine, loop_factory=practice_new_loop).result()

practice_servers = {"history": {"command": sys.executable, "args": [str(practice_server_path)], "transport": "stdio"}}

async def practice_fetch():
    return await MultiServerMCPClient(practice_servers).get_tools()

practice_mcp_tools = practice_wait(practice_fetch())
for practice_tool in practice_mcp_tools:
    print(f"  {practice_tool.name} · {type(practice_tool).__name__} · failure_type {practice_tool.args['failure_type'].get('enum')}")

**결과 해설** — 열린 Tool 은 `same_type_history` 하나이고, 타입은 LangChain 의 `StructuredTool` 입니다. 안에서 만든 Tool 과 똑같이 Agent 에 넣을 수 있습니다. `delete_record` 는 목록에 없습니다.

### 3단계 · 실패는 글자로 온다

In [ ]:
# 정상 인자와 없는 유형("XYZ")으로 각각 불러 봅니다.
# 안에서 만든 Tool 이었다면 없는 유형은 ValidationError 예외였습니다(Notebook 05).
# 관찰 포인트: MCP 를 거치면 서버 쪽 오류가 예외가 아니라 **정상 결과의 글자**로 돌아옵니다.
import json

def practice_text(result):
    return "".join(block.get("text", "") for block in result) if isinstance(result, list) else str(result)

def practice_succeeded(result):
    try:
        return "records" in json.loads(practice_text(result))
    except json.JSONDecodeError:
        return False

async def practice_call(args):
    return await practice_mcp_tools[0].ainvoke(args)

practice_good = practice_wait(practice_call({"failure_type": "HDF", "before": "2025-08-22T06:30:00+09:00"}))
practice_bad = practice_wait(practice_call({"failure_type": "XYZ", "before": "2025-08-22T06:30:00+09:00"}))
print("정상 :", practice_text(practice_good)[:60], "→ 성공으로 셈:", practice_succeeded(practice_good))
print("잘못 :", practice_text(practice_bad)[:60], "→ 성공으로 셈:", practice_succeeded(practice_bad))

assert {t.name for t in practice_mcp_tools} == {"same_type_history"}, "읽기 전용 목록 밖의 Tool 이 열렸습니다"
assert practice_mcp_tools[0].args["failure_type"]["enum"] == ["TWF", "HDF", "PWF", "OSF"]
assert practice_succeeded(practice_good)
assert "Error" in practice_text(practice_bad) and not practice_succeeded(practice_bad), "오류 글자를 성공으로 셌습니다"

**결과 해설** — 정상 호출은 기록이 담긴 JSON 이고, 없는 유형은 `Error executing tool …` 으로 시작하는 **글자**입니다. 예외가 나지 않았으므로 "예외가 없으면 성공"으로 세면 실패를 성공으로 셉니다. 그래서 결과에 기대한 키(`records`)가 있는지로 성공을 판단합니다.

## 직접 해 볼 과제
서버 코드의 `READ_ONLY` 에 "delete_record" 를 더해 서버 파일을 다시 쓰고, 2단계부터 다시 실행해 보세요. 무엇이 열립니까? 실제 앱은 이런 실수를 어떻게 막는지 `mcp_server.py` 와 비교해 보세요.

In [ ]:
# 읽기 전용 목록을 바꿔 서버 파일을 다시 씁니다. 실행 후 2단계 셀을 다시 돌리세요.
# 실제 앱은 목록 밖 Tool 을 등록하려 하면 서버가 아예 뜨지 않게 막습니다.
practice_try_read_only = '("same_type_history",)'
practice_server_path.write_text(practice_server_code.replace('("same_type_history",)', practice_try_read_only), encoding="utf-8")
print("서버 파일의 읽기 전용 목록:", practice_try_read_only)

## 중간 결과
서버가 정의한 함수와 실제로 연 Tool, 받아 온 Tool 의 타입과 enum, 그리고 정상 호출과 잘못된 호출이 각각 어떤 모양으로 돌아오는지 확인합니다.

## 실패 경계
MCP 서버가 뜨지 않으면 Tool 목록이 비어 옵니다. 그것을 "Tool 이 없다"로 넘기면 Agent 가 아무것도 못 하는 이유를 끝까지 모릅니다. 또 MCP 의 시간 초과 취소는 서버까지 가지 않습니다. 요청이 이미 실렸으면 서버는 실행을 마치고, 재시도하면 같은 Tool 이 두 번 돕니다. **쓰기 Tool 을 열지 않는 것**이 업무 데이터가 두 번 바뀌지 않는 유일한 근거입니다.

## 실제 app 연결
과제 10 App 의 `mcp_server.py::server` 가 `tools.py` 의 읽기 전용 목록에서 Tool 을 가져와 등록하고, 목록 밖 Tool 을 등록하려 하면 뜨지 않습니다. `MCP_MODE=on` 이면 `lookup.py::lookup_tools` 가 `shared/tools/mcp.py::mcp_tools` 로 이 서버를 자식 Process 로 띄워 Tool 을 받습니다. `lookup.py::succeeded` 가 결과 모양까지 보고 성공을 판단합니다.

## 다음 Notebook 연결
다음 `08_draft_chain.ipynb` 에서는 모은 근거로 LLM 이 초안을 쓰고, 그 초안을 검사합니다.